In [ ]:
# ============================================================================
# BREAST CANCER CLASSIFICATION - PROGETTO COMPLETO
# ============================================================================

# --- INSTALLAZIONE DIPENDENZE ---
import sys
!{sys.executable} -m pip install -q ucimlrepo

# --- IMPORT LIBRERIE ---
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.model_selection import train_test_split
from sklearn import metrics
import graphviz
from sklearn.tree import export_graphviz
import numpy as np
import scipy.stats as st
from sklearn.metrics import accuracy_score, confusion_matrix, precision_score, recall_score, f1_score, ConfusionMatrixDisplay
import seaborn as sns
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import KFold
from mpl_toolkits.mplot3d import Axes3D
from matplotlib.colors import ListedColormap
from sklearn.svm import SVC
from matplotlib.lines import Line2D
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.metrics import classification_report
from sklearn.model_selection import LeaveOneOut
from IPython.display import display

# --- CARICAMENTO DATI ---
from ucimlrepo import fetch_ucirepo

breast_cancer_wisconsin_diagnostic = fetch_ucirepo(id=17)

# --- SEPARAZIONE FEATURES E TARGET ---
X = breast_cancer_wisconsin_diagnostic.data.features
Y = breast_cancer_wisconsin_diagnostic.data.targets

# --- SPLITTING (DIVISIONE TRAIN/TEST) ---
X_train, X_test, Y_train, Y_test = train_test_split(
    X, Y,
    test_size=0.2,
    random_state=42,
    stratify=Y
)

print(f"Train Set: {X_train.shape}")
print(f"Test Set:  {X_test.shape}")
print("\nPrime righe del dataset:")
display(X.head())

# --- ENCODING NUMERICO TARGET (per consistenza tra tutti i modelli) ---
Y_train_num = Y_train['Diagnosis'].map({'M': 1, 'B': 0})
Y_test_num = Y_test['Diagnosis'].map({'M': 1, 'B': 0})
print(f"\nTarget codificato: M=1 (Maligno), B=0 (Benigno)")
print(f"Distribuzione Train: {Y_train_num.value_counts().to_dict()}")
print(f"Distribuzione Test: {Y_test_num.value_counts().to_dict()}")

# ============================================================================
# ANALISI ESPLORATIVA
# ============================================================================

# --- ANALISI DISTRIBUZIONE CLASSI ---
conteggi = Y_train["Diagnosis"].value_counts()
colori = ['red' if diagnosi == 'M' else 'blue' for diagnosi in conteggi.index]
plt.figure(figsize=(6, 5))
barre = plt.bar(conteggi.index, conteggi.values, color=colori, alpha=0.8, edgecolor='black')
plt.xlabel("Diagnosi")
plt.ylabel("Numero di Pazienti (Training Set)")
plt.title("Distribuzione Classi nel Training Set")
for rect in barre:
    height = rect.get_height()
    plt.text(rect.get_x() + rect.get_width()/2.0, height, f'{int(height)}', ha='center', va='bottom')
plt.grid(axis='y', linestyle='--', alpha=0.5)
plt.show()

totale = sum(conteggi)
print(f"Totale campioni nel Train: {totale}")
for diagnosi, count in conteggi.items():
    print(f"Classe {diagnosi}: {count} ({count/totale:.2%})")

# --- ANALISI CORRELAZIONI ---
plt.figure(figsize=(12, 10))
cols_to_plot = X_train.columns[:10]
corr_matrix = X_train[cols_to_plot].corr()
sns.heatmap(corr_matrix, annot=True, fmt=".2f", cmap='coolwarm', vmin=-1, vmax=1, square=True, linewidths=0.5)
plt.title('Matrice di Correlazione (Feature Medie - Training Set)')
plt.show()

# --- BOXPLOTS FEATURE ---
df_eda = X_train.copy()
df_eda['diagnosis'] = Y_train['Diagnosis'].values
features_to_plot = X_train.columns
num_plots = len(features_to_plot)
cols = 5
rows = (num_plots // cols) + (1 if num_plots % cols > 0 else 0)
plt.figure(figsize=(20, 4 * rows))
for i, col in enumerate(features_to_plot):
    plt.subplot(rows, cols, i+1)
    sns.boxplot(x='diagnosis', y=col, hue='diagnosis', data=df_eda, palette={'M': 'red', 'B': 'blue'}, legend=False)
    plt.title(col, fontsize=10, fontweight='bold')
    plt.xlabel('')
    plt.ylabel('')
plt.tight_layout()
plt.suptitle("Capacità Discriminante delle Feature (Training Set)", y=1.005, fontsize=16)
plt.show()

# --- ANALISI VARIABILE DISCRIMINANTE ---
feature_target = 'perimeter3'
df_temp = X_train[[feature_target]].copy()
df_temp['Diagnosis'] = Y_train['Diagnosis'].values
media_benigni = df_temp[df_temp['Diagnosis'] == 'B'][feature_target].mean()
media_maligni = df_temp[df_temp['Diagnosis'] == 'M'][feature_target].mean()
soglia_calcolata = (media_benigni + media_maligni) / 2
SOGLIA = int(round(soglia_calcolata))

print(f"Media Benigni ({feature_target}): {media_benigni:.2f}")
print(f"Media Maligni ({feature_target}): {media_maligni:.2f}")
print(f"Soglia discriminante calcolata: {SOGLIA}")

plt.figure(figsize=(10, 6))
plt.hist(df_temp[df_temp['Diagnosis']=='B'][feature_target], bins=30, alpha=0.6, color='blue', label='Benigni (Train)')
plt.hist(df_temp[df_temp['Diagnosis']=='M'][feature_target], bins=30, alpha=0.6, color='red', label='Maligni (Train)')
plt.axvline(SOGLIA, color='black', linestyle='dashed', linewidth=2, label=f'Soglia ({SOGLIA})')
plt.title(f'Distribuzione {feature_target}: Benigni vs Maligni (Training Set)')
plt.xlabel('Valore (es. mm)')
plt.ylabel('Numero di Pazienti')
plt.legend()
plt.grid(axis='y', alpha=0.3)
plt.show()

# --- ANALISI DETTAGLIATA (Basso vs Alto) ---
soglia = X_train[feature_target].mean()
df_plot = X_train[[feature_target]].copy()
df_plot['diagnosis'] = Y_train['Diagnosis'].values
df_plot['classe_dimensionale'] = pd.cut(df_plot[feature_target], bins=[-1, soglia, float('inf')], 
                                         labels=[f'Basso (< {soglia:.1f})', f'Alto (> {soglia:.1f})'])
grouped = df_plot.groupby(['classe_dimensionale', 'diagnosis'], observed=False).size().unstack(fill_value=0)
if 'B' in grouped.columns and 'M' in grouped.columns:
    grouped = grouped[['B', 'M']]

ax = grouped.plot(kind='bar', stacked=True, color=['blue', 'red'], figsize=(10, 6), width=0.6)
plt.title(f'Distribuzione Diagnosi basata su {feature_target}\n(Feature discriminante osservata nei Boxplot)', fontsize=14)
plt.xlabel(f'Valore di {feature_target}', fontsize=12)
plt.ylabel('Numero di Pazienti (Training Set)', fontsize=12)
plt.xticks(rotation=0)
plt.legend(title='Diagnosi', labels=['Benigno', 'Maligno'], loc='upper right')
plt.grid(axis='y', linestyle='--', alpha=0.5)
for p in ax.patches:
    width, height = p.get_width(), p.get_height()
    x, y = p.get_xy()
    if height > 0:
        ax.text(x + width/2, y + height/2, f'{int(height)}', ha='center', va='center', color='white', fontweight='bold')
plt.tight_layout()
plt.show()
print(f"Soglia utilizzata (Media del Train): {soglia:.2f}")

# ============================================================================
# PCA (PRINCIPAL COMPONENT ANALYSIS)
# ============================================================================

# --- PREPARAZIONE PCA (SCALING) ---
features = list(X_train.columns)
scaler = StandardScaler()
scaled_data_train = scaler.fit_transform(X_train[features])

# --- ADDESTRAMENTO PCA TEMPORANEO ---
pca_temp = PCA().fit(scaled_data_train)

# --- GRAFICO SCREE PLOT ---
plt.plot(range(1, pca_temp.n_components_ + 1), pca_temp.explained_variance_ratio_, marker='o')
plt.xlabel('Componenti della PCA')
plt.ylabel('Varianza spiegata')
plt.title("Risultati della PCA")
plt.show()

# --- ANALISI AUTOVALORI (Kaiser Rule) ---
eigenvalues = pca_temp.explained_variance_
n_samples = scaled_data_train.shape[0]
cov_matrix = np.dot(scaled_data_train.T, scaled_data_train) / (n_samples - 1)

print(f"--- Analisi Autovalori (Regola di Kaiser: Cerchiamo valori > 1) ---")
for i, eigenvalue in enumerate(eigenvalues):
    giudizio = "Utile (>1)" if eigenvalue >= 1 else "Debole (<1)"
    print(f"Componente {i+1}: {eigenvalue:.4f} -> {giudizio}")
print(f"\nSomma totale autovalori: {np.sum(eigenvalues):.2f} (Dovrebbe essere ~30)")

# --- ANALISI VARIANZA CUMULATA ---
cumulative_variance = np.cumsum(pca_temp.explained_variance_ratio_)
print("--- Varianza spiegata cumulata (prime 10 componenti) ---")
for i, var in enumerate(cumulative_variance[:10]):
    print(f"Componenti: {i+1} -> Varianza spiegata cumulata: {var:.2%}")

plt.figure(figsize=(10, 6))
plt.plot(range(1, len(cumulative_variance)+1), cumulative_variance, marker='o', linestyle='--', color='blue')
plt.axhline(y=0.95, color='r', linestyle='-', label='Soglia 95% Info')
plt.xlabel('Numero di Componenti')
plt.ylabel('Varianza Cumulata')
plt.title('Quante componenti servono per arrivare al 95%?', fontsize=14)
plt.legend(loc='lower right')
plt.grid(True, alpha=0.3)
plt.show()

# --- TRASFORMAZIONE EFFETTIVA (PCA DEFINITIVA) ---
n_comp = 6
pca_final = PCA(n_components=n_comp)
X_train_pca = pca_final.fit_transform(scaled_data_train)
scaled_data_test = scaler.transform(X_test[features])
X_test_pca = pca_final.transform(scaled_data_test)
cols_pca = [f'PC{i+1}' for i in range(n_comp)]

X_train_pca_df = pd.DataFrame(data=X_train_pca, columns=cols_pca, index=X_train.index)
X_test_pca_df = pd.DataFrame(data=X_test_pca, columns=cols_pca, index=X_test.index)
dataset_pca_train = pd.concat([X_train_pca_df, Y_train], axis=1)

print(f"Dati di Train trasformati: {X_train_pca.shape}")
print(f"Dati di Test trasformati:  {X_test_pca.shape}")
print("\nPrime 5 righe del nuovo dataset trasformato (Training):")
display(X_train_pca_df.head())

# --- VISUALIZZAZIONE SCATTER PLOT 2D ---
label_to_color = {"B": "blue", "M": "red"}
fig, ax = plt.subplots(figsize=(8, 6))
for label, color in label_to_color.items():
    mask = (Y_train["Diagnosis"] == label).values
    ax.scatter(X_train_pca[mask, 0], X_train_pca[mask, 1], color=color, 
               label="Benigno" if label == "B" else "Maligno", alpha=0.7, edgecolors='w')
ax.set_xlabel(f'Componente 1 ({pca_final.explained_variance_ratio_[0]:.1%} varianza)')
ax.set_ylabel(f'Componente 2 ({pca_final.explained_variance_ratio_[1]:.1%} varianza)')
ax.set_title('PCA: Distribuzione Maligni vs Benigni nello spazio PC1-PC2')
ax.legend()
ax.grid(True, linestyle='--', alpha=0.5)
plt.show()

# --- ANALISI LOADINGS ---
loadings = pca_final.components_
features = X_train.columns
num_pc = n_comp
pc_list = [f'PC{i+1}' for i in range(num_pc)]
loadings_df = pd.DataFrame(loadings.T, columns=pc_list, index=features)

plt.figure(figsize=(18, 14))
ax = sns.heatmap(loadings_df, annot=True, fmt=".2f", cmap='coolwarm', center=0, linewidths=0.5,
                 annot_kws={"size": 8}, cbar_kws={"orientation": "vertical", "label": "Peso (Loading)"})
plt.title("Matrice dei Loadings: Quali feature creano le Componenti?", fontsize=16)
plt.ylabel("Variabili Originali", fontsize=12)
plt.xlabel("Componenti Principali (PC)", fontsize=12)
plt.tight_layout()
plt.show()

# --- RANKING FEATURE IMPORTANCE ---
n_components_kept = pca_final.n_components_
loadings_sq = pca_final.components_ ** 2
variance_ratios = pca_final.explained_variance_ratio_
weighted_importance = np.dot(loadings_sq.T, variance_ratios)

feature_importance = pd.DataFrame({'Variabile': features, 'Punteggio_Importanza': weighted_importance})
feature_importance['Contributo_Perc'] = (feature_importance['Punteggio_Importanza'] / feature_importance['Punteggio_Importanza'].sum()) * 100
feature_importance = feature_importance.sort_values(by='Contributo_Perc', ascending=False)

plt.figure(figsize=(10, 8))
sns.barplot(x='Contributo_Perc', y='Variabile', hue='Variabile', data=feature_importance, palette='viridis', legend=False)
plt.axvline(x=100/len(features), color='r', linestyle='--', label='Soglia Media')
plt.title(f'Ranking (Basato sulle top {n_components_kept} componenti)')
plt.show()
print(feature_importance.head(10))

# --- PREPARAZIONE DATASET FINALE ---
n_components_reali = X_train_pca.shape[1]
colonne_pca = [f'PC{i+1}' for i in range(n_components_reali)]
df_pca = pd.DataFrame(data=X_train_pca, columns=colonne_pca)
df_pca['Diagnosis'] = Y_train['Diagnosis'].values
print(f"Dimensioni DataFrame: {df_pca.shape}")
print("-" * 30)
display(df_pca.head())

n_componenti_reali = X_train_pca.shape[1]
X_train_final = pd.DataFrame(data=X_train_pca, columns=[f'PC{i+1}' for i in range(n_componenti_reali)])
X_test_final = pd.DataFrame(data=X_test_pca, columns=[f'PC{i+1}' for i in range(n_componenti_reali)])
X_cv = X_train_final.reset_index(drop=True)

print("MODIFICA EFFETTUATA CON SUCCESSO")
print(f"Il modello ora userà {X_cv.shape[1]} variabili: {list(X_cv.columns)}")
display(X_cv.head())

# ============================================================================
# ALBERO DI DECISIONE
# ============================================================================

# --- PREPARAZIONE DATI ---
if 'X_train_pca' not in locals():
    print("ERRORE: Non trovo 'X_train_pca'. Assicurati di aver eseguito la cella della PCA!")
else:
    print(f"1. Dati PCA trovati: {X_train_pca.shape[1]} componenti per {X_train_pca.shape[0]} pazienti.")

print("2. Target (y) normalizzato e allineato.")
print("3. Indici resettati correttamente (da 0 a N).")
print("-" * 40)

# --- K-FOLD CROSS-VALIDATION ---
n_fold = 10
folds = StratifiedKFold(n_splits=n_fold, shuffle=True, random_state=42)
X_cv = X_train_final.reset_index(drop=True)
y_cv_num = Y_train_num.reset_index(drop=True)  # USO ENCODING NUMERICO

accuracy_k_fold = []
precision_k_fold = []
recall_k_fold = []
f1_k_fold = []
global_confusion_matrix = None

for n_fold_idx, (train_idx, valid_idx) in enumerate(folds.split(X_cv, y_cv_num)):
    X_train_fold, X_valid_fold = X_cv.iloc[train_idx], X_cv.iloc[valid_idx]
    y_train_fold, y_valid_fold = y_cv_num.iloc[train_idx], y_cv_num.iloc[valid_idx]
    
    model = DecisionTreeClassifier(max_depth=3, random_state=42)
    model.fit(X_train_fold, y_train_fold)
    
    plt.figure(figsize=(16, 8))
    plot_tree(model, filled=True, feature_names=X_cv.columns, class_names=['Benigno', 'Maligno'], 
              rounded=True, fontsize=10)
    plt.title(f"Albero Decisionale - Fold {n_fold_idx+1}")
    plt.show()
    
    y_pred_valid = model.predict(X_valid_fold)
    acc = accuracy_score(y_valid_fold, y_pred_valid)
    prec = precision_score(y_valid_fold, y_pred_valid, pos_label=1, zero_division=0)
    rec = recall_score(y_valid_fold, y_pred_valid, pos_label=1, zero_division=0)
    f1 = f1_score(y_valid_fold, y_pred_valid, pos_label=1, zero_division=0)
    
    accuracy_k_fold.append(acc)
    precision_k_fold.append(prec)
    recall_k_fold.append(rec)
    f1_k_fold.append(f1)
    
    cm = confusion_matrix(y_valid_fold, y_pred_valid, labels=[0, 1])
    if global_confusion_matrix is None:
        global_confusion_matrix = cm
    else:
        global_confusion_matrix += cm
    
    print(f"Fold {n_fold_idx+1}: Acc={acc:.1%} | F1={f1:.1%}")

mean_acc = np.mean(accuracy_k_fold)
conf_interval = st.t.interval(confidence=0.95, df=len(accuracy_k_fold)-1, loc=mean_acc, scale=st.sem(accuracy_k_fold))

print("\n" + "="*40)
print(f"Accuratezza Media: {mean_acc:.2%}")
print(f"Intervallo Confidenza (95%): [{conf_interval[0]:.2%}, {conf_interval[1]:.2%}]")
print(f"F1-Score Medio: {np.mean(f1_k_fold):.2%}")
print("="*40)

plt.figure(figsize=(6, 5))
sns.heatmap(global_confusion_matrix, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Pred: Benigno', 'Pred: Maligno'],
            yticklabels=['Reale: Benigno', 'Reale: Maligno'])
plt.title('Matrice di Confusione Globale (Somma dei 10 Fold)')
plt.show()

# --- LEAVE ONE OUT ---
X_loo_input = X_cv.copy()
y_loo_input = y_cv_num.copy()
X_array = X_loo_input.values if hasattr(X_loo_input, "values") else X_loo_input
y_array = y_loo_input.values if hasattr(y_loo_input, "values") else y_loo_input
if y_array.ndim > 1:
    y_array = y_array.ravel()

loo = LeaveOneOut()
y_true_all = []
y_pred_all = []
accuracy_history = []

for train_index, test_index in loo.split(X_array):
    X_train_loo, X_test_loo = X_array[train_index], X_array[test_index]
    y_train_loo, y_test_loo = y_array[train_index], y_array[test_index]
    
    model_loo = DecisionTreeClassifier(max_depth=3, random_state=42)
    model_loo.fit(X_train_loo, y_train_loo)
    pred = model_loo.predict(X_test_loo)
    
    y_true_all.append(y_test_loo[0])
    y_pred_all.append(pred[0])
    accuracy_history.append(1 if y_test_loo[0] == pred[0] else 0)

total_acc = accuracy_score(y_true_all, y_pred_all)
total_prec = precision_score(y_true_all, y_pred_all, pos_label=1, zero_division=0)
total_rec = recall_score(y_true_all, y_pred_all, pos_label=1, zero_division=0)
total_f1 = f1_score(y_true_all, y_pred_all, pos_label=1, zero_division=0)

conf_interval_loo = st.t.interval(confidence=0.95, df=len(accuracy_history)-1, 
                                   loc=np.mean(accuracy_history), scale=st.sem(accuracy_history))

print("\n" + "="*40)
print(f"Accuratezza Globale: {total_acc:.2%} (IC 95%: [{conf_interval_loo[0]:.2%}, {conf_interval_loo[1]:.2%}])")
print(f"F1-Score Globale:    {total_f1:.2%}")
print("="*40)

cm_global_loo = confusion_matrix(y_true_all, y_pred_all, labels=[0, 1])
plt.figure(figsize=(6, 5))
sns.heatmap(cm_global_loo, annot=True, fmt='d', cmap='Greens',
            xticklabels=['Pred: Benigno', 'Pred: Maligno'],
            yticklabels=['Reale: Benigno', 'Reale: Maligno'])
plt.title(f'Matrice di Confusione LOO (Somma di {len(X_array)} test individuali)')
plt.show()

# --- PRUNING ---
X_train_pruning = X_train_final.copy()
X_test_pruning = X_test_final.copy()
y_train_pruning = Y_train_num.reset_index(drop=True)
y_test_pruning = Y_test_num.reset_index(drop=True)

print(f"Dati caricati per il Pruning. Feature usate: {list(X_train_pruning.columns)}")
print("Inizio calcolo del pruning path...")
print("STRATEGIA: Creo un albero MOLTO profondo, poi lo poto usando Cross-Validation\n")

# Creo un albero SENZA limiti di profondità (max overfitting)
clf_overfitted = DecisionTreeClassifier(random_state=42, min_samples_split=2, min_samples_leaf=1)
clf_overfitted.fit(X_train_pruning, y_train_pruning)
print(f"Albero iniziale: Profondità={clf_overfitted.get_depth()}, Foglie={clf_overfitted.get_n_leaves()}")

path = clf_overfitted.cost_complexity_pruning_path(X_train_pruning, y_train_pruning)
ccp_alphas = path.ccp_alphas
print(f"Individuati {len(ccp_alphas)} livelli di potatura possibili.")
print(f"Range Alpha: da {ccp_alphas[0]:.6f} a {ccp_alphas[-1]:.6f}\n")

# Valuto ogni alpha con 5-Fold Cross-Validation
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_scores_mean = []
train_scores = []
test_scores = []

print("Valutazione Alpha con Cross-Validation (5-Fold):")
for i, ccp_alpha in enumerate(ccp_alphas[:-1]):  # Escludo l'ultimo (albero vuoto)
    clf = DecisionTreeClassifier(random_state=42, ccp_alpha=ccp_alpha)
    
    # Cross-validation sul training set
    cv_scores = cross_val_score(clf, X_train_pruning, y_train_pruning, cv=cv, scoring='accuracy')
    cv_scores_mean.append(cv_scores.mean())
    
    # Scores su train e test completi (per il grafico)
    clf.fit(X_train_pruning, y_train_pruning)
    train_scores.append(clf.score(X_train_pruning, y_train_pruning))
    test_scores.append(clf.score(X_test_pruning, y_test_pruning))
    
    if i % 10 == 0:  # Stampo ogni 10 alpha
        print(f"  Alpha {ccp_alpha:.6f}: CV={cv_scores.mean():.4f}, Test={test_scores[-1]:.4f}")

# Scelgo l'alpha con il miglior CV score
best_index = np.argmax(cv_scores_mean)
best_alpha = ccp_alphas[best_index]
print(f"\n✓ Miglior Alpha (basato su CV): {best_alpha:.6f}")
print(f"  CV Accuracy: {cv_scores_mean[best_index]:.4f}")
print(f"  Test Accuracy: {test_scores[best_index]:.4f}")

# Grafico Train vs Test vs CV
plt.figure(figsize=(12, 6))
plt.plot(ccp_alphas[:-1], train_scores, marker='o', label="Train", drawstyle="steps-post", alpha=0.7)
plt.plot(ccp_alphas[:-1], cv_scores_mean, marker='s', label="Cross-Validation (5-Fold)", drawstyle="steps-post", linewidth=2)
plt.plot(ccp_alphas[:-1], test_scores, marker='^', label="Test", drawstyle="steps-post", alpha=0.7)
plt.axvline(best_alpha, color='red', linestyle='--', label=f'Alpha ottimale={best_alpha:.4f}')
plt.xlabel("Alpha (Penalità complessità)")
plt.ylabel("Accuratezza")
plt.title("Ricerca dell'Alpha ottimale con Cross-Validation")
plt.legend()
plt.grid(True, linestyle='--', alpha=0.6)
plt.xscale('log')  # Scala logaritmica per vedere meglio i dettagli
plt.show()

# Diagnostica
print(f"\nAccuratezza Train con alpha={best_alpha:.6f}: {train_scores[best_index]:.4f}")
print(f"Differenza (Train - CV): {train_scores[best_index] - cv_scores_mean[best_index]:.4f}")
if train_scores[best_index] - cv_scores_mean[best_index] < 0.05:
    print("✓ Buona generalizzazione: Train e CV sono simili")
else:
    print("⚠ Possibile overfitting: Train superiore a CV")

prunedModel = DecisionTreeClassifier(random_state=42, ccp_alpha=best_alpha)
prunedModel.fit(X_train_pruning, y_train_pruning)

print(f"\nProfondità albero finale: {prunedModel.get_depth()}")
print(f"Numero di foglie: {prunedModel.get_n_leaves()}")

plt.figure(figsize=(30, 15))
plot_tree(prunedModel, filled=True, feature_names=list(X_train_pruning.columns), 
          class_names=['Benigno', 'Maligno'], rounded=True, fontsize=12, precision=2)
plt.title(f"Albero di Decisione Finale Potato (Alpha={best_alpha:.4f})", fontsize=25)
plt.tight_layout()
plt.show()

# --- VALUTAZIONE ALBERO POTATO SUL TEST SET ---
y_pred_pruned = prunedModel.predict(X_test_pruning)
acc_pruned = accuracy_score(y_test_pruning, y_pred_pruned)
cm_pruned = confusion_matrix(y_test_pruning, y_pred_pruned, labels=[0, 1])
recall_pruned = cm_pruned[1,1] / (cm_pruned[1,0] + cm_pruned[1,1]) if (cm_pruned[1,0] + cm_pruned[1,1]) > 0 else 0
f1_pruned = f1_score(y_test_pruning, y_pred_pruned, pos_label=1)

print("\n" + "="*40)
print("VALUTAZIONE ALBERO POTATO SUL TEST SET")
print("="*40)
print(f"Accuratezza Test: {acc_pruned:.2%}")
print(f"Recall Maligno: {recall_pruned:.2%}")
print(f"F1-Score: {f1_pruned:.2%}")
print("="*40)

plt.figure(figsize=(6, 5))
sns.heatmap(cm_pruned, annot=True, fmt='d', cmap='Purples',
            xticklabels=['Pred: Benigno', 'Pred: Maligno'],
            yticklabels=['Reale: Benigno', 'Reale: Maligno'])
plt.title('Matrice di Confusione - Albero Potato (Test Set)')
plt.show()

# --- VALUTAZIONE DECISION TREE BASE (max_depth=3) SUL TEST SET ---
print("\n" + "="*40)
print("VALUTAZIONE DECISION TREE BASE SUL TEST SET")
print("="*40)

dt_base = DecisionTreeClassifier(max_depth=3, random_state=42)
dt_base.fit(X_train_final, y_cv_num)

y_pred_dt_base = dt_base.predict(X_test_final)
y_test_final_num = Y_test_num.reset_index(drop=True)

acc_dt_base = accuracy_score(y_test_final_num, y_pred_dt_base)
cm_dt_base = confusion_matrix(y_test_final_num, y_pred_dt_base, labels=[0, 1])
recall_dt_base = cm_dt_base[1,1] / (cm_dt_base[1,0] + cm_dt_base[1,1]) if (cm_dt_base[1,0] + cm_dt_base[1,1]) > 0 else 0
f1_dt_base = f1_score(y_test_final_num, y_pred_dt_base, pos_label=1)

print(f"Accuratezza Test: {acc_dt_base:.2%}")
print(f"Recall Maligno: {recall_dt_base:.2%}")
print(f"F1-Score: {f1_dt_base:.2%}")
print("="*40)

plt.figure(figsize=(6, 5))
sns.heatmap(cm_dt_base, annot=True, fmt='d', cmap='Oranges',
            xticklabels=['Pred: Benigno', 'Pred: Maligno'],
            yticklabels=['Reale: Benigno', 'Reale: Maligno'])
plt.title('Matrice di Confusione - DT Base (max_depth=3, Test Set)')
plt.show()

# ============================================================================
# SUPPORT VECTOR MACHINE
# ============================================================================

# --- PREPARAZIONE DATI SVM ---
if 'X_train_final' in locals() and 'X_test_final' in locals():
    X_svm_train = X_train_final.copy()
    X_svm_test = X_test_final.copy()
else:
    cols = [f'PC{i+1}' for i in range(X_train_pca.shape[1])]
    X_svm_train = pd.DataFrame(X_train_pca, columns=cols)
    X_svm_test = pd.DataFrame(X_test_pca, columns=cols)

y_train_num_svm = Y_train_num.reset_index(drop=True)
y_test_num_svm = Y_test_num.reset_index(drop=True)

X_svm_train = X_svm_train.reset_index(drop=True)
X_svm_test = X_svm_test.reset_index(drop=True)

# --- VISUALIZZAZIONE GEOMETRICA 2D ---
X_viz = X_svm_train.iloc[:, :2].values
y_viz = y_train_num_svm.values

models_viz = [
    (SVC(kernel='linear', C=1.0), 'Kernel Lineare'),
    (SVC(kernel='rbf', gamma='auto', C=1.0), 'Kernel RBF (Gaussiano)'),
    (SVC(kernel='poly', degree=3, C=1.0), 'Kernel Polinomiale')
]

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
plt.subplots_adjust(wspace=0.25)

for (clf, title), ax in zip(models_viz, axes.flatten()):
    clf.fit(X_viz, y_viz)
    x_min, x_max = X_viz[:, 0].min() - 1, X_viz[:, 0].max() + 1
    y_min, y_max = X_viz[:, 1].min() - 1, X_viz[:, 1].max() + 1
    xx, yy = np.meshgrid(np.linspace(x_min, x_max, 200), np.linspace(y_min, y_max, 200))
    Z = clf.decision_function(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
    
    ax.pcolormesh(xx, yy, Z > 0, cmap=ListedColormap(['#AAAAFF', '#FFAAAA']), shading='auto', alpha=0.4)
    ax.contour(xx, yy, Z, colors='k', levels=[-1, 0, 1], linestyles=['--', '-', '--'], linewidths=[1, 2, 1])
    ax.scatter(X_viz[:, 0], X_viz[:, 1], c=y_viz, cmap=ListedColormap(['#0000FF', '#FF0000']), edgecolor='k', s=30)
    ax.set_title(title, fontweight='bold')
    ax.set_xlabel("PC1 (Dimensione/Varianza Max)")
    ax.set_ylabel("PC2 (Forma/Varianza Secondaria)")

plt.show()

# --- CROSS VALIDATION SVM ---
models_eval = [
    ('SVM Linear', SVC(kernel='linear', C=1.0)),
    ('SVM RBF', SVC(kernel='rbf', gamma='scale', C=1.0)),
    ('SVM Poly', SVC(kernel='poly', degree=3, C=1.0))
]

cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

print("Risultati Cross-Validation (Accuracy su 10 Fold):")
print("-" * 60)

for name, model in models_eval:
    cv_scores = cross_val_score(model, X_svm_train, y_train_num_svm, cv=cv, scoring='accuracy')
    print(f"{name:<15} | Media: {cv_scores.mean():.2%} | Stabilità (Std): {cv_scores.std():.2%}")

# --- TEST SET EVALUATION ---
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for (name, model), ax in zip(models_eval, axes.flatten()):
    model.fit(X_svm_train, y_train_num_svm)
    y_pred = model.predict(X_svm_test)
    acc = accuracy_score(y_test_num_svm, y_pred)
    cm = confusion_matrix(y_test_num_svm, y_pred)
    recall = cm[1,1] / (cm[1,0] + cm[1,1]) if (cm[1,0] + cm[1,1]) > 0 else 0
    
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Benigno', 'Maligno'])
    disp.plot(cmap='Blues', ax=ax, colorbar=False)
    ax.set_title(f"{name}\nAcc: {acc:.1%} | Recall: {recall:.1%}")

plt.tight_layout()
plt.show()

# Salviamo le predizioni SVM per la tabella finale
svm_predictions = {}
for name, model in models_eval:
    svm_predictions[name] = model.predict(X_svm_test)

# --- KERNEL TRICK 3D ---
print("Generazione Visualizzazione 3D (Kernel Trick)...")

x_min, x_max = X_viz[:, 0].min() - 1, X_viz[:, 0].max() + 1
y_min, y_max = X_viz[:, 1].min() - 1, X_viz[:, 1].max() + 1
xx_3d, yy_3d = np.meshgrid(np.linspace(x_min, x_max, 50), np.linspace(y_min, y_max, 50))

svm_3d = SVC(kernel='rbf', gamma=0.5, C=1.0)
svm_3d.fit(X_viz, y_viz)

Z_surf = svm_3d.decision_function(np.c_[xx_3d.ravel(), yy_3d.ravel()]).reshape(xx_3d.shape)
Z_points = svm_3d.decision_function(X_viz)

fig = plt.figure(figsize=(12, 10))
ax = fig.add_subplot(111, projection='3d')

surf = ax.plot_surface(xx_3d, yy_3d, Z_surf, cmap='coolwarm', alpha=0.6, linewidth=0, antialiased=True)
ax.contour(xx_3d, yy_3d, Z_surf, levels=[0], colors='black', linewidths=3, linestyles='solid')

ax.scatter(X_viz[y_viz==0, 0], X_viz[y_viz==0, 1], Z_points[y_viz==0],
           c='blue', s=30, edgecolor='k', alpha=0.8, label='Benigno (Valle)')
ax.scatter(X_viz[y_viz==1, 0], X_viz[y_viz==1, 1], Z_points[y_viz==1],
           c='red', s=30, edgecolor='k', alpha=0.8, label='Maligno (Picco)')

ax.view_init(elev=30, azim=-60)
ax.set_title("Il 'Kernel Trick' svelato:\nL'SVM proietta i dati in 3D (Z) per separarli linearmente", fontsize=14)
ax.set_xlabel("PC 1")
ax.set_ylabel("PC 2")
ax.set_zlabel("Distanza dal Margine (Z)")
plt.legend()
plt.tight_layout()
plt.show()

# ============================================================================
# TABELLA RIASSUNTIVA FINALE
# ============================================================================

summary_dict = {
    'Modello': ['Decision Tree (max_depth=3)', 'Decision Tree Potato', 'SVM Linear', 'SVM RBF', 'SVM Poly'],
    'Accuratezza Test': [
        acc_dt_base,
        acc_pruned,
        accuracy_score(y_test_num_svm, svm_predictions['SVM Linear']),
        accuracy_score(y_test_num_svm, svm_predictions['SVM RBF']),
        accuracy_score(y_test_num_svm, svm_predictions['SVM Poly'])
    ],
    'Recall Maligno': [
        recall_dt_base,
        recall_pruned,
        confusion_matrix(y_test_num_svm, svm_predictions['SVM Linear'])[1,1] /
        max(1, confusion_matrix(y_test_num_svm, svm_predictions['SVM Linear'])[1,0] +
            confusion_matrix(y_test_num_svm, svm_predictions['SVM Linear'])[1,1]),
        confusion_matrix(y_test_num_svm, svm_predictions['SVM RBF'])[1,1] /
        max(1, confusion_matrix(y_test_num_svm, svm_predictions['SVM RBF'])[1,0] +
            confusion_matrix(y_test_num_svm, svm_predictions['SVM RBF'])[1,1]),
        confusion_matrix(y_test_num_svm, svm_predictions['SVM Poly'])[1,1] /
        max(1, confusion_matrix(y_test_num_svm, svm_predictions['SVM Poly'])[1,0] +
            confusion_matrix(y_test_num_svm, svm_predictions['SVM Poly'])[1,1])
    ],
    'F1-Score': [
        f1_dt_base,
        f1_pruned,
        f1_score(y_test_num_svm, svm_predictions['SVM Linear']),
        f1_score(y_test_num_svm, svm_predictions['SVM RBF']),
        f1_score(y_test_num_svm, svm_predictions['SVM Poly'])
    ],
    'Note': [
        'Test Set',
        f'Pruned (alpha={best_alpha:.6f})',
        'Test Set',
        'Test Set',
        'Test Set'
    ]
}

summary_df = pd.DataFrame(summary_dict)
summary_df[['Accuratezza Test', 'Recall Maligno', 'F1-Score']] = summary_df[['Accuratezza Test', 'Recall Maligno', 'F1-Score']].applymap(lambda x: f"{x:.2%}")

print("\n" + "="*70)
print("=== TABELLA RIASSUNTIVA MODELLI ===")
print("="*70)
print("\nNOTA: TUTTI i modelli sono valutati sul Test Set finale")
print("      Encoding consistente: 0=Benigno, 1=Maligno")
print(f"      K-Fold CV media (riferimento): Acc={mean_acc:.2%}, F1={np.mean(f1_k_fold):.2%}\n")
display(summary_df)
print("="*70)
print("ANALISI COMPLETATA - NESSUN DATA LEAKAGE!")
print("="*70)